# Input File Generation

In [ ]:
import json
from tqdm import tqdm


PROMPT_TEMPLATE = """You are given a tweet text:

{text}

Your task is to briefly explain factual background knowledge about the entities in the text. (person, organization, location, ...) mentioned in the tweet.

Guidelines:
- Output 1–2 short sentences (≤ 50 words) that provide factual information about entities from the text.
- Be objective and concise. Do not over explain.
- Only explain entity spans, not other words. 
- Use the tokens exactly as given in the input. 
- Do NOT change casing, spacing, or spelling. Copy them verbatim.
- If unsure or no knowledge is available, output nothing.
"""


def parse_bio_file(bio_path):
    """
    BIO 형식 txt → {IMGID: [token1, token2, ...]} 형태 변환
    - IMGID 라인 robust 처리
    - 빈 줄은 문장 구분이므로 skip
    - token만 추출 (BIO, POS 등이 있어도 첫 컬럼만)
    """
    data = {}
    current_imgid = None
    current_tokens = []

    with open(bio_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()

           
            if line.startswith("IMGID:"):
                
                if current_imgid is not None and current_tokens:
                    data[current_imgid] = current_tokens

                # New IMGID
                temp = line.split("IMGID:")[1].strip()
                current_imgid = temp.split()[0]     # 공백/탭 제거
                current_tokens = []
                continue

            
            if not line:
                continue

            # extract tokens only
            parts = line.split()
            if len(parts) > 0:
                token = parts[0]
                current_tokens.append(token)

    # save the last block
    if current_imgid is not None and current_tokens:
        data[current_imgid] = current_tokens

    return data


# --- Generating batch JSONL ---
def make_batch_jsonl(bio_path, output_jsonl_path):
    data = parse_bio_file(bio_path)

    with open(output_jsonl_path, "w", encoding="utf-8") as outfile:

        for imgid, tokens in tqdm(data.items(), desc="Building batch input"):

            
            text = " ".join(tokens)

            user_prompt = PROMPT_TEMPLATE.format(text=text)

            # Generate Batch API payload
            payload = {
                "custom_id": f"{imgid}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": "gpt-4o-mini",
                    "temperature": 0.2,
                    "max_tokens": 128,
                    "messages": [
                        {"role": "user", "content": user_prompt}
                    ]
                }
            }

            outfile.write(json.dumps(payload, ensure_ascii=False) + "\n")

    print(f"✅ Generated Batch input: {output_jsonl_path}")



if __name__ == "__main__":
    bio_input = r""
    batch_output = r""
    
    make_batch_jsonl(bio_input, batch_output)


# Calling API

In [ ]:
#### batch create 
from openai import OpenAI
client = OpenAI(api_key=API_KEY)
# available only in version after openai==1.2.0
batch_input_file = client.files.create(
  file=open(r"", "rb"),
  purpose="batch"
)

batch_input_file_id = batch_input_file.id

client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h", #응답 요청 시간(240502기준 24h만 가능)
    metadata={
      "description": "mner knowledge twit 2015 test"
    }
)

# Saving Results

In [ ]:
import json
from tqdm import tqdm


def parse_bio_file(bio_path):
    data = {}
    current_imgid = None
    current_tokens = []

    with open(bio_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()

            if line.startswith("IMGID:"):
                if current_imgid is not None and current_tokens:
                    data[current_imgid] = current_tokens
                temp = line.split("IMGID:")[1].strip()
                current_imgid = temp.split()[0]
                current_tokens = []
                continue

            if not line:
                continue

            parts = line.split()
            if len(parts) > 0:
                token = parts[0]
                current_tokens.append(token)

    if current_imgid is not None and current_tokens:
        data[current_imgid] = current_tokens

    return data


# --- batch output(JSONL) → imgid → knowledge mapping ---
def load_knowledge_map(batch_output_jsonl):
    knowledge_map = {}

    with open(batch_output_jsonl, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Reading batch output"):
            obj = json.loads(line)

            custom_id = obj.get("custom_id", "")
            if not custom_id:
                continue

            imgid = custom_id.strip()

            try:
                content = obj["response"]["body"]["choices"][0]["message"]["content"].strip()
            except Exception:
                content = ""

            knowledge_map[imgid] = content

    return knowledge_map


# --- BIO + knowledge → final JSONL ---
def make_final_jsonl(bio_path, batch_output_path, final_output_path):
    print("📌 Reading the BIO...")
    bio_data = parse_bio_file(bio_path)

    print("📌 Mapping knowledge from the batch output...")
    knowledge_map = load_knowledge_map(batch_output_path)

    print("📌 Writing the final JSONL file...")
    with open(final_output_path, "w", encoding="utf-8") as outfile:
        for imgid, tokens in tqdm(bio_data.items(), desc="Writing merged JSONL"):

            text = " ".join(tokens)
            knowledge = knowledge_map.get(imgid, "")

            out_obj = {
                "imgid": imgid,
                "text": text,
                "knowledge": knowledge
            }

            outfile.write(json.dumps(out_obj, ensure_ascii=False) + "\n")

    print(f"Completed! Final JSONL generated → {final_output_path}")



if __name__ == "__main__":
    bio_input = ""
    batch_output = r""
    final_output = r""

    make_final_jsonl(bio_input, batch_output, final_output)
